<a href="https://colab.research.google.com/github/akshat-646/Stock-Prediction-Models/blob/main/LSTM_Stock_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Install Dependencies

In [1]:
!pip install -q yfinance scikit-learn plotly

## 2. Imports & Device Setup

In [2]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import yfinance as yf
import plotly.graph_objects as go

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from datetime import timedelta

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


## 3. Data Pipeline (`data_pipeline.py`)

In [4]:
ticker_to_id = {t: i for i, t in enumerate(TICKERS)}

class DataPipeline:
    def __init__(self):
        self.scalers  = {}   # one MinMaxScaler per ticker
        self.date_map = {}   # DatetimeIndex per ticker

    # ── Fetch raw OHLCV ──────────────────────────────────────────────
    def fetch_data(self, ticker):
        df = yf.download(ticker, period="10y", progress=False)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        df = df.rename(columns=str.lower)
        df = df[["open", "high", "low", "close", "volume"]]
        return df.dropna()

    # ── Technical indicators ─────────────────────────────────────────
    def add_features(self, df):
        df = df.copy()
        df["return_1d"]    = df["close"].pct_change()
        df["return_5d"]    = df["close"].pct_change(5)
        df["sma_20"]       = df["close"].rolling(20).mean()
        df["ema_12"]       = df["close"].ewm(span=12).mean()
        df["ema_26"]       = df["close"].ewm(span=26).mean()
        df["macd"]         = df["ema_12"] - df["ema_26"]
        delta = df["close"].diff()
        gain  = delta.clip(lower=0).rolling(14).mean()
        loss  = (-delta.clip(upper=0)).rolling(14).mean()
        rs    = gain / loss.replace(0, np.nan)
        df["rsi"]          = 100 - (100 / (1 + rs))
        std = df["close"].rolling(20).std()
        df["bb_width"]     = (2 * std) / df["sma_20"]
        df["volatility"]   = df["return_1d"].rolling(20).std()
        df["volume_ratio"] = df["volume"] / df["volume"].rolling(20).mean()
        df = df.replace([np.inf, -np.inf], np.nan)
        return df.dropna()

    # ── Sliding-window sequences ─────────────────────────────────────
    def create_sequences(self, data, sid):
        X, y, s = [], [], []
        for i in range(len(data) - INPUT_LEN - OUTPUT_LEN):
            X.append(data[i:i + INPUT_LEN])
            y.append(data[i + INPUT_LEN:i + INPUT_LEN + OUTPUT_LEN, 3])
            s.append(sid)
        return np.array(X), np.array(y), np.array(s)

    # ── Build combined train / test from ALL tickers ─────────────────
    def prepare(self):
        X_tr, y_tr, s_tr = [], [], []
        X_te, y_te, s_te = [], [], []
        for ticker in TICKERS:
            df     = self.fetch_data(ticker)
            df     = self.add_features(df)
            self.date_map[ticker] = df.index
            scaler = MinMaxScaler()
            scaled = scaler.fit_transform(df.values)
            self.scalers[ticker] = scaler
            X, y, s = self.create_sequences(scaled, ticker_to_id[ticker])
            split = int(len(X) * 0.7)
            X_tr.extend(X[:split]); y_tr.extend(y[:split]); s_tr.extend(s[:split])
            X_te.extend(X[split:]); y_te.extend(y[split:]); s_te.extend(s[split:])
        return {
            "train": (np.array(X_tr), np.array(y_tr), np.array(s_tr)),
            "test":  (np.array(X_te), np.array(y_te), np.array(s_te))
        }

    # ── Prepare a single (possibly unseen) ticker ────────────────────
    def prepare_single(self, ticker, fit_scaler=True, scaler=None):
        df = self.fetch_data(ticker)
        df = self.add_features(df)
        self.date_map[ticker] = df.index
        if fit_scaler:
            scaler = MinMaxScaler()
            scaler.fit(df.values)
        self.scalers[ticker] = scaler
        scaled = scaler.transform(df.values)
        X, y, s = self.create_sequences(scaled, sid=0)
        split   = int(len(X) * 0.7)
        return (
            X[:split], y[:split],
            X[split:], y[split:],
            df.index
        )

## 4. Model Definition (`model.py`)

In [5]:
class LSTMModel(nn.Module):
    """
    LSTM model with an optional per-stock embedding.

    Architecture
    ─────────────
    Input  →  Linear projection (input_size → hidden_size)
           +  Stock embedding  (optional, shape: hidden_size)
           →  LSTM (num_layers)
           →  Last hidden state
           →  FC  (hidden_size → output_len)

    The stock embedding is added to every timestep of the projected
    input, letting the model learn stock-specific biases while sharing
    all LSTM weights across tickers — exactly the same idea as the
    Transformer version's positional + stock embedding.
    """
    def __init__(self, input_size, num_stocks,
                 hidden_size=HIDDEN_SIZE, num_layers=NUM_LAYERS, output_len=OUTPUT_LEN):
        super().__init__()
        self.use_embedding = num_stocks > 1
        # Project raw features to hidden space
        self.proj  = nn.Linear(input_size, hidden_size)
        # Learnable stock embedding (one vector per ticker)
        if self.use_embedding:
            self.embed = nn.Embedding(num_stocks, hidden_size)
        # Core LSTM (operates in hidden_size space)
        self.lstm  = nn.LSTM(hidden_size, hidden_size, num_layers,
                             batch_first=True, dropout=0.2)
        self.fc    = nn.Linear(hidden_size, output_len)

    def forward(self, x, sid=None):
        # x: (B, T, input_size)
        x = self.proj(x)                                   # → (B, T, H)
        if self.use_embedding and sid is not None:
            x = x + self.embed(sid).unsqueeze(1)           # broadcast over T
        out, _ = self.lstm(x)                              # → (B, T, H)
        return self.fc(out[:, -1, :])

## 5. Configuration

In [3]:
INPUT_LEN   = 90
OUTPUT_LEN  = 5
EPOCHS      = 25
BATCH_SIZE  = 64
HIDDEN_SIZE = 128
NUM_LAYERS  = 2

TICKERS = [
    "RELIANCE.NS","TCS.NS","HDFCBANK.NS","ICICIBANK.NS","KOTAKBANK.NS",
    "SBIN.NS","AXISBANK.NS","BAJFINANCE.NS","BAJAJFINSV.NS","HDFCLIFE.NS",
    "SBILIFE.NS","ITC.NS","HINDUNILVR.NS","NESTLEIND.NS","TATACONSUM.NS",
    "BRITANNIA.NS","ASIANPAINT.NS","ULTRACEMCO.NS","GRASIM.NS","LT.NS",
    "ADANIENT.NS","ADANIPORTS.NS","BHARTIARTL.NS","INDUSINDBK.NS","MARUTI.NS",
    "HEROMOTOCO.NS","EICHERMOT.NS","BAJAJ-AUTO.NS","POWERGRID.NS","NTPC.NS",
    "ONGC.NS","COALINDIA.NS","JSWSTEEL.NS","TATASTEEL.NS","HCLTECH.NS",
    "WIPRO.NS","TECHM.NS","INFY.NS","CIPLA.NS","DRREDDY.NS",
    "APOLLOHOSP.NS","SUNPHARMA.NS","DIVISLAB.NS","LTIM.NS","TITAN.NS",
    "INDIGO.NS","TRENT.NS","JIOFIN.NS"
]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


## 6. Train Unified LSTM (`train_unified_lstm.py`)

In [6]:
pipeline   = DataPipeline()
MODEL_PATH = "nifty50_lstm_model.pt"

if os.path.exists(MODEL_PATH):
    print(f"Loading model from {MODEL_PATH} ...")
    checkpoint  = torch.load(MODEL_PATH, map_location=device, weights_only=False)
    INPUT_FEATS = checkpoint["input_size"]
    model       = LSTMModel(INPUT_FEATS, len(TICKERS)).to(device)
    model.load_state_dict(checkpoint["model_state"])
    print(f"Model loaded  (input_size={INPUT_FEATS})")

    if "scalers" in checkpoint:
        pipeline.scalers  = checkpoint["scalers"]
        pipeline.date_map = checkpoint["date_map"]
        print("  Scalers loaded from checkpoint")
    else:
        print("  No scalers in checkpoint — re-fetching training data for scalers.")
        splits      = pipeline.prepare()
        INPUT_FEATS = splits["train"][0].shape[2]

else:
    print("No checkpoint found — training from scratch (this will take a while).")
    print("  Upload your .pt file to skip this step next time.")

    splits  = pipeline.prepare()
    X_train, y_train, s_train = splits["train"]
    INPUT_FEATS = X_train.shape[2]

    X_tr = torch.tensor(X_train, dtype=torch.float32)
    y_tr = torch.tensor(y_train, dtype=torch.float32)
    s_tr = torch.tensor(s_train, dtype=torch.long)
    loader = DataLoader(TensorDataset(X_tr, y_tr, s_tr),
                        batch_size=BATCH_SIZE, shuffle=True)

    model     = LSTMModel(INPUT_FEATS, len(TICKERS)).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0003)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3)
    loss_fn   = nn.MSELoss()

    for epoch in range(EPOCHS):
        model.train()
        total = 0
        for xb, yb, sb in loader:
            xb, yb, sb = xb.to(device), yb.to(device), sb.to(device)
            pred = model(xb, sb)
            loss = loss_fn(pred, yb)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total += loss.item()
        avg = total / len(loader)
        scheduler.step(avg)
        print(f"Epoch {epoch+1}/{EPOCHS}: loss={avg:.4f}")

    # Save checkpoint (model + scalers) for next time
    torch.save({
        "model_state": model.state_dict(),
        "input_size":  INPUT_FEATS,
        "scalers":     pipeline.scalers,
        "date_map":    pipeline.date_map,
    }, MODEL_PATH)
    print(f"Model saved to {MODEL_PATH}")

model.eval()
print("Model ready")

No checkpoint found — training from scratch (this will take a while).
  Upload your .pt file to skip this step next time.


/tmp/ipykernel_5514/2005058945.py:10: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period="10y", progress=False)
/tmp/ipykernel_5514/2005058945.py:10: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period="10y", progress=False)
/tmp/ipykernel_5514/2005058945.py:10: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period="10y", progress=False)
/tmp/ipykernel_5514/2005058945.py:10: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period="10y", progress=False)
/tmp/ipykernel_5514/2005058945.py:10: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period="10y", progress=False)
/tmp/ipykernel_5514/2005058945.py:10: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(

Epoch 1/25: loss=0.0026
Epoch 2/25: loss=0.0006
Epoch 3/25: loss=0.0005
Epoch 4/25: loss=0.0004
Epoch 5/25: loss=0.0004
Epoch 6/25: loss=0.0004
Epoch 7/25: loss=0.0004
Epoch 8/25: loss=0.0004
Epoch 9/25: loss=0.0003
Epoch 10/25: loss=0.0003
Epoch 11/25: loss=0.0003
Epoch 12/25: loss=0.0003
Epoch 13/25: loss=0.0003
Epoch 14/25: loss=0.0003
Epoch 15/25: loss=0.0003
Epoch 16/25: loss=0.0003
Epoch 17/25: loss=0.0003
Epoch 18/25: loss=0.0003
Epoch 19/25: loss=0.0003
Epoch 20/25: loss=0.0003
Epoch 21/25: loss=0.0003
Epoch 22/25: loss=0.0003
Epoch 23/25: loss=0.0003
Epoch 24/25: loss=0.0003
Epoch 25/25: loss=0.0003
Model saved to nifty50_lstm_model.pt
Model ready


In [7]:
from google.colab import files
files.download("nifty50_lstm_model.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## **FETCH & PREPARE THE UNSEEN STOCK**

In [8]:
UNSEEN_TICKER = input("Stock Name (e.g. PIDILITIND.NS): ")
print(f"Fetching {UNSEEN_TICKER} ...")

(
    X_unseen_train, y_unseen_train,
    X_unseen_test,  y_unseen_test,
    unseen_dates
) = pipeline.prepare_single(UNSEEN_TICKER)

print(f"{UNSEEN_TICKER} | Train seqs: {len(X_unseen_train)} | Test seqs: {len(X_unseen_test)}")
print(f"  Features per timestep: {X_unseen_test.shape[2]} (expected {INPUT_FEATS})")

assert X_unseen_test.shape[2] == INPUT_FEATS, \
    f"Feature mismatch! Got {X_unseen_test.shape[2]}, expected {INPUT_FEATS}"

Stock Name (e.g. PIDILITIND.NS): DMART.NS
Fetching DMART.NS ...
DMART.NS | Train seqs: 1480 | Test seqs: 635
  Features per timestep: 15 (expected 15)


/tmp/ipykernel_5514/2005058945.py:10: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period="10y", progress=False)


## **HELPERS: INVERSE TRANSFORM & INFERENCE**

In [9]:
def inverse_close(arr, ticker):
    """Inverse-scale a 1-D array of close prices back to ₹ values."""
    scaler = pipeline.scalers[ticker]
    dummy  = np.zeros((len(arr), INPUT_FEATS))
    dummy[:, 3] = arr          # column 3 = close
    return scaler.inverse_transform(dummy)[:, 3]


def run_inference(model_to_use, X, emb_override=None):
    """
    Run forward pass for every sample in X.

    emb_override: a (1, hidden_size) tensor used instead of the
                  learned embedding — used for zero-shot / few-shot.
    """
    model_to_use.eval()
    preds = []
    with torch.no_grad():
        for i in range(len(X)):
            x = torch.tensor(X[i], dtype=torch.float32).unsqueeze(0).to(device)
            if emb_override is not None:
                # Manually apply embedding override through the LSTM layers
                xp      = model_to_use.proj(x)               # (1, T, H)
                xp      = xp + emb_override.unsqueeze(1).to(device)
                out, _  = model_to_use.lstm(xp)
                out     = model_to_use.fc(out[:, -1, :])
            else:
                out = model_to_use(x)
            preds.append(out.cpu().numpy()[0])
    return np.array(preds)

## **FEW-SHOT ADAPTATION**

In [11]:
import copy

print("FEW-SHOT ADAPTATION")

fs_model= copy.deepcopy(model)
FEWSHOT_EPOCHS = 8
FEWSHOT_LR     = 0.001
new_emb_weight = torch.zeros(1, HIDDEN_SIZE)

if model.use_embedding:
    mean_emb       = model.embed.weight.mean(dim=0, keepdim=True)
    new_emb_weight = mean_emb.cpu().detach().clone()

# Learnable embedding vector for the unseen ticker
unseen_emb = nn.Parameter(new_emb_weight.to(device), requires_grad=True)

# Freeze all model parameters — only the new embedding will be updated
for param in fs_model.parameters():
    param.requires_grad = False

fs_optimizer = torch.optim.Adam([unseen_emb], lr=FEWSHOT_LR)
fs_loss_fn   = nn.MSELoss()

X_fs = torch.tensor(X_unseen_train, dtype=torch.float32)
y_fs = torch.tensor(y_unseen_train, dtype=torch.float32)
fs_loader = DataLoader(TensorDataset(X_fs, y_fs), batch_size=32, shuffle=True)

fs_model.train()
print(f"\nTraining new embedding on {len(X_fs)} sequences for {FEWSHOT_EPOCHS} epochs ...")

for epoch in range(FEWSHOT_EPOCHS):
    total = 0
    for xb, yb in fs_loader:
        xb, yb = xb.to(device), yb.to(device)
        xp       = fs_model.proj(xb)
        xp       = xp + unseen_emb.unsqueeze(1).expand(xb.size(0), -1, -1)
        out, _   = fs_model.lstm(xp)
        pred     = fs_model.fc(out[:, -1, :])
        loss     = fs_loss_fn(pred, yb)
        fs_optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_([unseen_emb], 1.0)
        fs_optimizer.step()
        total += loss.item()
    print(f"  Epoch {epoch+1}/{FEWSHOT_EPOCHS} | loss={total/len(fs_loader):.5f}")

# Evaluate few-shot model on test set
fs_preds_scaled = run_inference(fs_model, X_unseen_test,
                                emb_override=unseen_emb.detach())

fs_actual = inverse_close(y_unseen_test[:, 0], UNSEEN_TICKER)
fs_pred   = inverse_close(fs_preds_scaled[:, 0], UNSEEN_TICKER)

"""## **FEW-SHOT METRICS**"""

fs_rmse = np.sqrt(mean_squared_error(fs_actual, fs_pred))
fs_mae  = mean_absolute_error(fs_actual, fs_pred)
fs_r2   = r2_score(fs_actual, fs_pred)
fs_mape = np.mean(np.abs((fs_actual - fs_pred) / fs_actual)) * 100

print(f"\nFew-Shot Metrics — {UNSEEN_TICKER}")
print(f"  MAPE  : {fs_mape:.2f}%")
print(f"  RMSE  : ₹{fs_rmse:.2f}")
print(f"  MAE   : ₹{fs_mae:.2f}")
print(f"  R²    : {fs_r2:.4f}")

FEW-SHOT ADAPTATION

Training new embedding on 1480 sequences for 8 epochs ...


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/rnn.py:1141: UserWarning: RNN module weights are not part of single contiguous chunk of memory. This means they need to be compacted at every call, possibly greatly increasing memory usage. To compact weights again call flatten_parameters(). (Triggered internally at /pytorch/aten/src/ATen/native/cudnn/RNN.cpp:1480.)
  result = _VF.lstm(


  Epoch 1/8 | loss=0.00068
  Epoch 2/8 | loss=0.00064
  Epoch 3/8 | loss=0.00063
  Epoch 4/8 | loss=0.00061
  Epoch 5/8 | loss=0.00059
  Epoch 6/8 | loss=0.00057
  Epoch 7/8 | loss=0.00058
  Epoch 8/8 | loss=0.00056

Few-Shot Metrics — DMART.NS
  MAPE  : 1.29%
  RMSE  : ₹75.84
  MAE   : ₹54.54
  R²    : 0.9729


## **FUTURE FORECAST / PREDICTION**

In [12]:
print(f"Forecasting next {OUTPUT_LEN} trading days for {UNSEEN_TICKER} ...\n")

last_seq = torch.tensor(X_unseen_test[-1], dtype=torch.float32).unsqueeze(0).to(device)

fs_model.eval()
with torch.no_grad():
    xp            = fs_model.proj(last_seq)
    xp            = xp + unseen_emb.detach().unsqueeze(1)
    out, _        = fs_model.lstm(xp)
    fut_scaled    = fs_model.fc(out[:, -1, :]).cpu().numpy()[0]

fut_prices   = inverse_close(fut_scaled, UNSEEN_TICKER)
last_date    = unseen_dates[-1]
future_dates = [last_date + timedelta(days=i + 1) for i in range(OUTPUT_LEN)]

print(f"Last known close: ₹{fs_actual[-1]:.2f}  (date: {last_date.date()})\n")
for d, p in zip(future_dates, fut_prices):
    change = ((p - fs_actual[-1]) / fs_actual[-1]) * 100
    print(f"  {d.date()}  →  ₹{p:.2f}  ({change:+.2f}%)")

# ── Plotly chart ─────────────────────────────────────────────────────
N_plot_hist = 30

plot_actual_dates     = unseen_dates[-N_plot_hist:]
plot_actual_prices    = fs_actual[-N_plot_hist:]
plot_predicted_prices = fs_pred[-N_plot_hist:]

forecast_start_date      = unseen_dates[-1]
forecast_start_price     = fs_pred[-1]
forecast_dates_combined  = [forecast_start_date] + future_dates
forecast_prices_combined = [forecast_start_price] + list(fut_prices)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=plot_actual_dates, y=plot_actual_prices,
    name="Actual Prices", mode="lines",
    line=dict(color="#00E5FF", width=2)
))

fig.add_trace(go.Scatter(
    x=plot_actual_dates, y=plot_predicted_prices,
    name="Few-Shot Predicted Prices", mode="lines",
    line=dict(color="#69FF47", width=2)
))

fig.add_trace(go.Scatter(
    x=forecast_dates_combined, y=forecast_prices_combined,
    name=f"{OUTPUT_LEN}-Day Future Forecast", mode="lines",
    line=dict(color="#FFD700", width=2, dash="dot"),
    marker=dict(size=6, symbol="circle")
))

fig.update_layout(
    title=f"{UNSEEN_TICKER} — Last {N_plot_hist} Days & {OUTPUT_LEN}-Day Forecast",
    xaxis_title="Date",
    yaxis_title="Close Price (₹)",
    template="plotly_dark",
    legend=dict(x=1.02, y=1, xanchor="left", yanchor="top")
)
fig.show()


Forecasting next 5 trading days for DMART.NS ...

Last known close: ₹3770.80  (date: 2026-03-30)

  2026-03-31  →  ₹3803.95  (+0.88%)
  2026-04-01  →  ₹3794.83  (+0.64%)
  2026-04-02  →  ₹3823.61  (+1.40%)
  2026-04-03  →  ₹3832.89  (+1.65%)
  2026-04-04  →  ₹3803.69  (+0.87%)


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/rnn.py:1141: UserWarning: RNN module weights are not part of single contiguous chunk of memory. This means they need to be compacted at every call, possibly greatly increasing memory usage. To compact weights again call flatten_parameters(). (Triggered internally at /pytorch/aten/src/ATen/native/cudnn/RNN.cpp:1480.)
  result = _VF.lstm(


## **MULTI UNSEEN STOCK PREDICTION**

In [19]:
# 1. Define a list of unseen tickers
UNSEEN_TICKERS_LIST = [
    "DMART.NS",
    "DABUR.NS",
    "INFY.BO",]

print(f"Starting predictions for multiple unseen tickers: {UNSEEN_TICKERS_LIST}\n")

all_metrics = {}
all_plots = []

for current_ticker in UNSEEN_TICKERS_LIST:
    print(f"Processing ticker: {current_ticker}\n" + "="*40 + "\n")

    # 2. Fetch and prepare data for the current unseen ticker
    print(f"Fetching {current_ticker} ...")
    (
        X_unseen_train, y_unseen_train,
        X_unseen_test,  y_unseen_test,
        unseen_dates
    ) = pipeline.prepare_single(current_ticker)

    print(f"{current_ticker} | Train seqs: {len(X_unseen_train)} | Test seqs: {len(X_unseen_test)}")
    print(f"  Features per timestep: {X_unseen_test.shape[2]} (expected {INPUT_FEATS})")

    assert X_unseen_test.shape[2] == INPUT_FEATS, \
        f"Feature mismatch for {current_ticker}! Got {X_unseen_test.shape[2]}, expected {INPUT_FEATS}"

    # 3. Few-shot Adaptation
    print("\nFEW-SHOT ADAPTATION")
    fs_model = copy.deepcopy(model)
    FEWSHOT_EPOCHS = 8
    FEWSHOT_LR     = 0.001
    new_emb_weight = torch.zeros(1, HIDDEN_SIZE)

    if model.use_embedding:
        # Use mean embedding from trained model as starting point
        mean_emb       = fs_model.embed.weight.mean(dim=0, keepdim=True)
        new_emb_weight = mean_emb.cpu().detach().clone()

    # Learnable embedding vector for the unseen ticker
    unseen_emb = nn.Parameter(new_emb_weight.to(device), requires_grad=True)

    # Freeze all model parameters — only the new embedding will be updated
    for param in fs_model.parameters():
        param.requires_grad = False

    fs_optimizer = torch.optim.Adam([unseen_emb], lr=FEWSHOT_LR)
    fs_loss_fn   = nn.MSELoss()

    X_fs = torch.tensor(X_unseen_train, dtype=torch.float32)
    y_fs = torch.tensor(y_unseen_train, dtype=torch.float32)
    fs_loader = DataLoader(TensorDataset(X_fs, y_fs), batch_size=32, shuffle=True)

    fs_model.train()
    print(f"Training new embedding on {len(X_fs)} sequences for {FEWSHOT_EPOCHS} epochs ...")

    for epoch in range(FEWSHOT_EPOCHS):
        total = 0
        for xb, yb in fs_loader:
            xb, yb = xb.to(device), yb.to(device)
            xp       = fs_model.proj(xb)
            xp       = xp + unseen_emb.unsqueeze(1).expand(xb.size(0), -1, -1)
            out, _   = fs_model.lstm(xp)
            pred     = fs_model.fc(out[:, -1, :])
            loss     = fs_loss_fn(pred, yb)
            fs_optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_([unseen_emb], 1.0)
            fs_optimizer.step()
            total += loss.item()
        print(f"  Epoch {epoch+1}/{FEWSHOT_EPOCHS} | loss={total/len(fs_loader):.5f}")

    # 4. Evaluate Few-Shot Model on Test Set
    fs_preds_scaled = run_inference(fs_model, X_unseen_test,
                                    emb_override=unseen_emb.detach())

    fs_actual = inverse_close(y_unseen_test[:, 0], current_ticker)
    fs_pred   = inverse_close(fs_preds_scaled[:, 0], current_ticker)

    fs_rmse = np.sqrt(mean_squared_error(fs_actual, fs_pred))
    fs_mae  = mean_absolute_error(fs_actual, fs_pred)
    fs_r2   = r2_score(fs_actual, fs_pred)
    fs_mape = np.mean(np.abs((fs_actual - fs_pred) / fs_actual)) * 100

    print(f"\nFew-Shot Metrics — {current_ticker}")
    print(f"  MAPE  : {fs_mape:.2f}%")
    print(f"  RMSE  : ₹{fs_rmse:.2f}")
    print(f"  MAE   : ₹{fs_mae:.2f}")
    print(f"  R²    : {fs_r2:.4f}")

    all_metrics[current_ticker] = {
        "MAPE": fs_mape,
        "RMSE": fs_rmse,
        "MAE": fs_mae,
        "R2": fs_r2
    }

    # 5. Future Forecast
    print(f"\nForecasting next {OUTPUT_LEN} trading days for {current_ticker} ...")

    last_seq = torch.tensor(X_unseen_test[-1], dtype=torch.float32).unsqueeze(0).to(device)

    fs_model.eval()
    with torch.no_grad():
        xp            = fs_model.proj(last_seq)
        xp            = xp + unseen_emb.detach().unsqueeze(1)
        out, _        = fs_model.lstm(xp)
        fut_scaled    = fs_model.fc(out[:, -1, :]).cpu().numpy()[0]

    fut_prices   = inverse_close(fut_scaled, current_ticker)
    last_date    = unseen_dates[-1]
    future_dates = [last_date + timedelta(days=i + 1) for i in range(OUTPUT_LEN)]

    print(f"Last known close: ₹{fs_actual[-1]:.2f}  (date: {last_date.date()})\n")
    for d, p in zip(future_dates, fut_prices):
        change = ((p - fs_actual[-1]) / fs_actual[-1]) * 100
        print(f"  {d.date()}  →  ₹{p:.2f}  ({change:+.2f}%)")

    # 6. Plotly chart
    N_plot_hist = 30 # Reusing the variable

    plot_actual_dates     = unseen_dates[-N_plot_hist:]
    plot_actual_prices    = fs_actual[-N_plot_hist:]
    plot_predicted_prices = fs_pred[-N_plot_hist:]

    forecast_start_date      = unseen_dates[-1]
    forecast_start_price     = fs_pred[-1]
    forecast_dates_combined  = [forecast_start_date] + future_dates
    forecast_prices_combined = [forecast_start_price] + list(fut_prices)

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=plot_actual_dates, y=plot_actual_prices,
        name="Actual Prices", mode="lines",
        line=dict(color="#00E5FF", width=2)
    ))

    fig.add_trace(go.Scatter(
        x=plot_actual_dates, y=plot_predicted_prices,
        name="Few-Shot Predicted Prices", mode="lines",
        line=dict(color="#69FF47", width=2)
    ))

    fig.add_trace(go.Scatter(
        x=forecast_dates_combined, y=forecast_prices_combined,
        name=f"{OUTPUT_LEN}-Day Future Forecast", mode="lines",
        line=dict(color="#FFD700", width=2, dash="dot"),
        marker=dict(size=6, symbol="circle")
    ))

    fig.update_layout(
        title=f"{current_ticker} — Last {N_plot_hist} Days & {OUTPUT_LEN}-Day Forecast",
        xaxis_title="Date",
        yaxis_title="Close Price (₹)",
        template="plotly_dark",
        legend=dict(x=1.02, y=1, xanchor="left", yanchor="top")
    )
    all_plots.append(fig) # Store figures to display later, or display immediately

    print("\n" + "="*40 + "\n") # Separator for next ticker

print("All predictions and evaluations complete.\n")

# Display all plots at the end
for i, fig in enumerate(all_plots):
    print(f"Displaying plot for {UNSEEN_TICKERS_LIST[i]}")
    fig.show()

print("\nSummary of Metrics:")
for ticker, metrics in all_metrics.items():
    print(f"  {ticker}:")
    for metric_name, value in metrics.items():
        if metric_name in ["MAPE"]:
            print(f"    {metric_name}: {value:.2f}%")
        elif metric_name in ["RMSE", "MAE"]:
            print(f"    {metric_name}: ₹{value:.2f}")
        else:
            print(f"    {metric_name}: {value:.4f}")

Starting predictions for multiple unseen tickers: ['DMART.NS', 'DABUR.NS', 'INFY.BO']

Processing ticker: DMART.NS

Fetching DMART.NS ...
DMART.NS | Train seqs: 1480 | Test seqs: 635
  Features per timestep: 15 (expected 15)

FEW-SHOT ADAPTATION
Training new embedding on 1480 sequences for 8 epochs ...


/tmp/ipykernel_5514/2005058945.py:10: FutureWarning:

YF.download() has changed argument auto_adjust default to True

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/rnn.py:1141: UserWarning:

RNN module weights are not part of single contiguous chunk of memory. This means they need to be compacted at every call, possibly greatly increasing memory usage. To compact weights again call flatten_parameters(). (Triggered internally at /pytorch/aten/src/ATen/native/cudnn/RNN.cpp:1480.)



  Epoch 1/8 | loss=0.00067
  Epoch 2/8 | loss=0.00066
  Epoch 3/8 | loss=0.00061
  Epoch 4/8 | loss=0.00060
  Epoch 5/8 | loss=0.00059
  Epoch 6/8 | loss=0.00057
  Epoch 7/8 | loss=0.00057
  Epoch 8/8 | loss=0.00056

Few-Shot Metrics — DMART.NS
  MAPE  : 1.32%
  RMSE  : ₹77.62
  MAE   : ₹55.79
  R²    : 0.9716

Forecasting next 5 trading days for DMART.NS ...
Last known close: ₹3770.80  (date: 2026-03-30)

  2026-03-31  →  ₹3797.49  (+0.71%)
  2026-04-01  →  ₹3788.49  (+0.47%)
  2026-04-02  →  ₹3817.53  (+1.24%)
  2026-04-03  →  ₹3826.99  (+1.49%)
  2026-04-04  →  ₹3798.09  (+0.72%)


Processing ticker: DABUR.NS

Fetching DABUR.NS ...


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/rnn.py:1141: UserWarning:

RNN module weights are not part of single contiguous chunk of memory. This means they need to be compacted at every call, possibly greatly increasing memory usage. To compact weights again call flatten_parameters(). (Triggered internally at /pytorch/aten/src/ATen/native/cudnn/RNN.cpp:1480.)

/tmp/ipykernel_5514/2005058945.py:10: FutureWarning:

YF.download() has changed argument auto_adjust default to True



DABUR.NS | Train seqs: 1649 | Test seqs: 707
  Features per timestep: 15 (expected 15)

FEW-SHOT ADAPTATION
Training new embedding on 1649 sequences for 8 epochs ...


...

  Epoch 1/8 | loss=0.00048
  Epoch 2/8 | loss=0.00034
  Epoch 3/8 | loss=0.00033
  Epoch 4/8 | loss=0.00032
  Epoch 5/8 | loss=0.00032
  Epoch 6/8 | loss=0.00033
  Epoch 7/8 | loss=0.00032
  Epoch 8/8 | loss=0.00032

Few-Shot Metrics — INFY.BO
  MAPE  : 1.32%
  RMSE  : ₹28.16
  MAE   : ₹20.53
  R²    : 0.9794

Forecasting next 5 trading days for INFY.BO ...
Last known close: ₹1254.60  (date: 2026-03-30)

  2026-03-31  →  ₹1221.94  (-2.60%)
  2026-04-01  →  ₹1219.21  (-2.82%)
  2026-04-02  →  ₹1224.27  (-2.42%)
  2026-04-03  →  ₹1229.86  (-1.97%)
  2026-04-04  →  ₹1220.16  (-2.75%)


All predictions and evaluations complete.

Displaying plot for DMART.NS


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/rnn.py:1141: UserWarning:

RNN module weights are not part of single contiguous chunk of memory. This means they need to be compacted at every call, possibly greatly increasing memory usage. To compact weights again call flatten_parameters(). (Triggered internally at /pytorch/aten/src/ATen/native/cudnn/RNN.cpp:1480.)



Displaying plot for DABUR.NS


Displaying plot for INFY.BO



Summary of Metrics:
  DMART.NS:
    MAPE: 1.32%
    RMSE: ₹77.62
    MAE: ₹55.79
    R2: 0.9716
  DABUR.NS:
    MAPE: 1.01%
    RMSE: ₹7.28
    MAE: ₹5.32
    R2: 0.9684
  INFY.BO:
    MAPE: 1.32%
    RMSE: ₹28.16
    MAE: ₹20.53
    R2: 0.9794
